# Credit Card Default Prediction

This notebook builds a baseline model that predicts whether a credit card client will **default on their payment next month**.

**Dataset:** [Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) (UCI Machine Learning Repository, id 350). It covers 30,000 clients in Taiwan from April to September 2005, with 23 features, no missing values and a binary target.

**Workflow:**
1. Load the data from the UCI repository
2. Inspect and clean the data, and engineer new features
3. Split it into train / validation / test sets
4. Build a preprocessing pipeline
5. Train and compare two linear models (Logistic Regression and Linear SVC)

## 1. Load the data

`fetch_ucirepo` downloads the dataset and returns the features (`X`) and the target (`y`) as pandas DataFrames, along with metadata and variable descriptions. The two are concatenated into a single DataFrame `df`.

The columns are named `X1`–`X23` and `Y`:

| Column | Meaning |
|---|---|
| `X1` | Credit limit (NT dollars), including individual and family/supplementary credit |
| `X2` | Gender (1 = male, 2 = female) |
| `X3` | Education (1 = graduate school, 2 = university, 3 = high school, 4 = others) |
| `X4` | Marital status (1 = married, 2 = single, 3 = others) |
| `X5` | Age (years) |
| `X6`–`X11` | Repayment status from September (`X6`) back to April (`X11`): -1 = paid on time, 1–9 = months of delay |
| `X12`–`X17` | Bill statement amount from September (`X12`) back to April (`X17`) |
| `X18`–`X23` | Amount paid from September (`X18`) back to April (`X23`) |
| `Y` | Default payment next month (1 = yes, 0 = no) |

The raw data also contains some codes the official documentation doesn't describe, such as -2 and 0 in the repayment status columns and 0, 5 and 6 in education. The cleaning step below deals with them.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
default_of_credit_card_clients = fetch_ucirepo(id=350) 
  
# data (as pandas dataframes) 
X = default_of_credit_card_clients.data.features 
y = default_of_credit_card_clients.data.targets 
  
# metadata 
print(default_of_credit_card_clients.metadata) 
  
# variable information 
print(default_of_credit_card_clients.variables) 

df = pd.concat([X, y], axis=1)

{'uci_id': 350, 'name': 'Default of Credit Card Clients', 'repository_url': 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients', 'data_url': 'https://archive.ics.uci.edu/static/public/350/data.csv', 'abstract': "This research aimed at the case of customers' default payments in Taiwan and compares the predictive accuracy of probability of default among six data mining methods.", 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 30000, 'num_features': 23, 'feature_types': ['Integer', 'Real'], 'demographics': ['Sex', 'Education Level', 'Marital Status', 'Age'], 'target_col': ['Y'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C55S3H', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 365, 'type': 'NATIVE', 'title': 'The comparisons of data mining techniques for the predictive accuracy of 

## 2. Data inspection, cleaning and feature engineering

This cell prepares the data for modelling.

**Target detection:** The code looks for the target column under several common names (`Y` here, while other copies of the dataset use `default.payment.next.month` or `default payment next month`). This lets the pipeline work with any of them. If none is found, it raises an error.

**Basic inspection:** The code checks for missing values, duplicate client IDs and the class balance of the target. Rows whose target isn't 0 or 1 are dropped. `ID` is removed from the features if present, because it carries no predictive information.

**Column groups:**
- `pay_cols` (`X6`–`X11`): monthly repayment status
- `bill_cols` (`X12`–`X17`): monthly bill amounts
- `payment_cols` (`X18`–`X23`): monthly payment amounts

### `engineer_features(X)`

This function is defined here and applied to each data split in the next step. It first replaces invalid or undocumented values:

| Column | Rule |
|---|---|
| `X5` (age) | Ages outside 18–100 → `NaN` (filled in later by the imputer) |
| `X1` (credit limit) | Limits ≤ 0 → `NaN` |
| `X2` (gender) | Anything other than 1 or 2 → `NaN` |
| `X3` (education) | Anything other than 1–4 → 0 ("unknown") |
| `X4` (marriage) | Anything other than 1–3 → 0 ("unknown") |

It then adds 9 features that summarise each client's behaviour over the six months:

| Feature | Definition | Intuition |
|---|---|---|
| `delinquent_months` | Number of months with repayment status > 0 | How often the client paid late |
| `max_delay` | Longest delay in the six months (statuses ≤ 0 count as 0) | How bad the worst late payment was |
| `severe_delay_months` | Number of months with a delay of 2 or more months | Persistent, serious delinquency |
| `avg_bill` | Mean monthly bill amount | Typical level of debt |
| `recent_bill_change` | September bill − August bill | Whether the debt grew recently |
| `avg_payment` | Mean monthly payment amount | Typical repayment capacity |
| `zero_payment_months` | Number of months with a payment of 0 | Months with no payment (this can include months when nothing was owed) |
| `credit_utilization` | September bill (negative bills set to 0) ÷ credit limit | How close the client is to their credit limit |
| `payment_to_bill_ratio` | Total paid ÷ total of positive bills, clipped to [0, 5]. `NaN` if the client had no bills | How much of the billed amount the client actually repaid |

Each feature uses only values from the client's own row, so the function learns nothing from the dataset as a whole. Applying it separately to each split therefore leaks no information between them.

### Observations from the output
- There are **no missing values** in any column.
- The duplicate-ID check prints nothing because `ucimlrepo` doesn't include the `ID` column in the features, so the `if` branch is skipped.
- The target is **imbalanced**: about 77.9% of clients did not default and 22.1% did. Accuracy alone is therefore misleading, because a model that always predicts "no default" would already score about 78%.

In [11]:
# Data pipiline


import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

data = df.copy()
data.columns = data.columns.str.strip()

# Detect target column
target = next(
    (c for c in [
        "Y",
        "default.payment.next.month",
        "default payment next month"
    ] if c in data.columns),
    None
)

if target is None:
    raise ValueError("Target column not found.")

# Basic inspection
print("Missing values:")
print(data.isna().sum())

print("\nDuplicate IDs:")
if "ID" in data.columns:
    print(data["ID"].duplicated().sum())

print("\nTarget distribution:")
print(data[target].value_counts(normalize=True))

# Remove rows with invalid or missing target labels
data = data[data[target].isin([0, 1])].copy()

y = data[target].astype(int)
X = data.drop(columns=[target, "ID"], errors="ignore")

pay_cols = [f"X{i}" for i in range(6, 12)]
bill_cols = [f"X{i}" for i in range(12, 18)]
payment_cols = [f"X{i}" for i in range(18, 24)]

def engineer_features(X):
    X = X.copy()

    # Age: mark implausible values as missing
    X["X5"] = X["X5"].where(
        X["X5"].between(18, 100), np.nan
    )

    # Invalid credit limits
    X["X1"] = X["X1"].where(X["X1"] > 0, np.nan)

    # Gender: preserve valid categories
    X["X2"] = X["X2"].where(
        X["X2"].isin([1, 2]), np.nan
    )

    # Education: 0 is reserved for unknown
    X["X3"] = X["X3"].where(
        X["X3"].isin([1, 2, 3, 4]), 0
    )

    # Marriage: 0 is reserved for unknown
    X["X4"] = X["X4"].where(
        X["X4"].isin([1, 2, 3]), 0
    )

    # Repayment behavior
    delays = X[pay_cols].clip(lower=0)

    X["delinquent_months"] = (
        X[pay_cols] > 0
    ).sum(axis=1)

    X["max_delay"] = delays.max(axis=1)

    X["severe_delay_months"] = (
        X[pay_cols] >= 2
    ).sum(axis=1)

    # Bill features
    X["avg_bill"] = X[bill_cols].mean(axis=1)

    X["recent_bill_change"] = (
        X["X12"] - X["X13"]
    )

    # Payment features
    X["avg_payment"] = X[payment_cols].mean(axis=1)

    X["zero_payment_months"] = (
        X[payment_cols] == 0
    ).sum(axis=1)

    # Credit utilization proxy
    X["credit_utilization"] = (
        X["X12"].clip(lower=0) / X["X1"]
    )

    # Payment-to-bill proxy
    positive_bills = X[bill_cols].clip(lower=0)
    total_bills = positive_bills.sum(axis=1)
    total_payments = X[payment_cols].sum(axis=1)

    X["payment_to_bill_ratio"] = (
        total_payments /
        total_bills.replace(0, np.nan)
    ).clip(lower=0, upper=5)

    return X


Missing values:
X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
X11    0
X12    0
X13    0
X14    0
X15    0
X16    0
X17    0
X18    0
X19    0
X20    0
X21    0
X22    0
X23    0
Y      0
dtype: int64

Duplicate IDs:

Target distribution:
Y
0    0.7788
1    0.2212
Name: proportion, dtype: float64


## 3. Train / validation / test split

The data is split in two steps:
1. 20% is held out as the **test set**. It is kept aside for the final evaluation only.
2. The remaining 80% is split 75 / 25 into a **training set** and a **validation set**.

This gives a **60 / 20 / 20** split overall. The validation set is used to compare models and tune choices, so the test set stays untouched until the end.

- `stratify` keeps the ~22% default rate the same in all three sets.
- `random_state=42` makes the split reproducible.

`engineer_features` is then applied to each set. Each set ends up with **32 columns**: the 23 original features plus the 9 engineered ones. The sets contain 18,000, 6,000 and 6,000 rows.

In [12]:

# 60% train, 20% validation, 20% test
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.25,
    stratify=y_dev,
    random_state=42
)

# Deterministic row-level engineering
X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (18000, 32)
Validation: (6000, 32)
Test: (6000, 32)


## 4. Preprocessing pipeline

The columns are split into two groups, and a `ColumnTransformer` processes each group differently.

### Categorical columns
Gender (`X2`), education (`X3`), marriage (`X4`) and the six repayment status columns (`X6`–`X11`).

The repayment status columns are treated as categories rather than numbers because their codes don't sit on one consistent scale. The values -2, -1 and 0 describe different kinds of non-late behaviour, while 1–9 count months of delay.

- `SimpleImputer(strategy="most_frequent")` fills missing values with the most common category. For example, this covers invalid gender values that were set to `NaN` during cleaning.
- `OneHotEncoder(handle_unknown="ignore")` creates one binary column per category. A category that appears in the validation or test data but not in training is encoded as all zeros instead of raising an error.

### Numerical columns
All the other columns, including the engineered features.

- `SimpleImputer(strategy="median", add_indicator=True)` fills missing values with the column median. For each feature that had missing values in training, it also adds a binary "was missing" column. An example is `payment_to_bill_ratio` for clients with no bills, where the fact that a value is missing can be informative in itself.
- `RobustScaler()` centres each feature on its median and scales it by the interquartile range. Bill and payment amounts are heavily skewed and contain large outliers, and this scaler is much less affected by them than standard scaling.

### No data leakage
The preprocessor is only *defined* here. It is fitted later, inside each model pipeline, on the training data only. Medians, categories and scaling ranges are therefore never learned from the validation or test sets.

In [13]:

categorical_cols = [
    "X2", "X3", "X4"
] + pay_cols

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])


## 5. Model training and evaluation

Two linear classifiers are compared:
- **Logistic Regression** models the probability of default directly. `max_iter=3000` gives the solver enough iterations to converge.
- **Linear SVC** is a linear support vector machine. It finds the boundary that separates the two classes with the widest margin. `max_iter=10000` is set for the same reason.

For each model, the preprocessor and the classifier are combined into a single `Pipeline`. `clone(preprocessor)` gives each model its own fresh, unfitted copy of the preprocessor. Each pipeline is fitted on the training set and evaluated on the validation set.

Linear SVC has no `predict_proba` method, so its `decision_function` scores are used instead. This works because ROC-AUC and PR-AUC depend only on how the scores rank the clients, not on whether the scores are probabilities.

### Metrics

| Metric | What it measures |
|---|---|
| Precision (class 1) | Of the clients predicted to default, the share who actually defaulted |
| Recall (class 1) | Of the clients who actually defaulted, the share the model caught |
| F1-score | The harmonic mean of precision and recall |
| ROC-AUC | The probability that a randomly chosen defaulter gets a higher score than a randomly chosen non-defaulter (0.5 = random guessing, 1.0 = perfect) |
| PR-AUC (Average Precision) | The area under the precision–recall curve. It is more informative than ROC-AUC on imbalanced data. A random model scores roughly the positive rate (~0.22) |
| Confusion matrix | `[[TN, FP], [FN, TP]]`: rows are the actual classes and columns are the predicted classes |

The fitted pipelines are stored in `fitted_models` for later use.

In [14]:

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        random_state=42
    ),
    "Linear SVC": LinearSVC(
        max_iter=10000,
        random_state=42
    )
}

fitted_models = {}

for name, estimator in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", estimator)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_val)[:, 1]
    else:
        y_score = pipeline.decision_function(X_val)

    print(f"\n{name}")
    print(classification_report(
        y_val, y_pred, zero_division=0
    ))

    print("ROC-AUC:", roc_auc_score(y_val, y_score))
    print("PR-AUC (AP):", average_precision_score(
        y_val, y_score
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_pred))

    fitted_models[name] = pipeline



Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4673
           1       0.69      0.35      0.47      1327

    accuracy                           0.82      6000
   macro avg       0.76      0.65      0.68      6000
weighted avg       0.80      0.82      0.80      6000

ROC-AUC: 0.7754516115038838
PR-AUC (AP): 0.5452047936779886
Confusion matrix:
[[4459  214]
 [ 860  467]]

Linear SVC
              precision    recall  f1-score   support

           0       0.84      0.96      0.89      4673
           1       0.68      0.34      0.46      1327

    accuracy                           0.82      6000
   macro avg       0.76      0.65      0.67      6000
weighted avg       0.80      0.82      0.80      6000

ROC-AUC: 0.7733613274223113
PR-AUC (AP): 0.5430685823415591
Confusion matrix:
[[4463  210]
 [ 872  455]]


### Results on the validation set

| Model | Accuracy | Precision (default) | Recall (default) | F1 (default) | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.82 | 0.69 | 0.35 | 0.47 | 0.775 | 0.545 |
| Linear SVC | 0.82 | 0.68 | 0.34 | 0.46 | 0.773 | 0.543 |

**What the results show:**
- **The two models perform almost the same.** Both are linear, so they find very similar decision boundaries.
- **Accuracy is misleading here.** 82% is only about 4 percentage points better than always predicting "no default" (~78%).
- **Recall on defaulters is low.** Logistic Regression catches 467 of the 1,327 defaulters (35%) and misses 860 of them. For a lender, a missed defaulter usually costs more than a false alarm, so this is the main weakness.
- **The models still separate the classes reasonably well.** A ROC-AUC of ~0.77 and a PR-AUC of ~0.54 (compared with a random baseline of ~0.22) show real ranking ability. Lowering the decision threshold would raise recall at the cost of precision.

**Possible next steps:**
- Use `class_weight="balanced"` so the models give more weight to the minority (default) class.
- Tune the decision threshold on the validation set to get a better balance of precision and recall.
- Try non-linear models (for example random forest or gradient boosting) that can capture interactions between features.
- Evaluate the final chosen model on the test set only once, at the very end.